# 03-性能分析流程：改善 memcpy 的带宽

本节使用一个 512 KB memcpy 小测试完成一次基于 XS-GEM5 平台的性能分析：

1. 关闭预取，观察 memcpy 带宽；
2. 用 Top-Down 计数器观察处理器瓶颈；
3. 使用 PerfCCT，可视化每一条指令的执行并抓取热点循环；
4. 开启 KMHv3 默认预取，对比带宽变化，确认性能收益来源。

## 1. 工作负载：512 KB memcpy

memcpy workload：将一段数据从某一块内存区域复制到另一块内存区域。

In [ ]:
%%bash
source ../env.sh
ASSET=$(get_asset workload)
ls -lh "$ASSET/gem5-memcpy-riscv64-xs.bin" "$ASSET/gem5-memcpy-riscv64-xs.txt"

## 2. 关闭预取的 kmhv3 GEM5 模型运行 memcpy

In [ ]:
%%bash
set -euo pipefail
source ../env.sh

RUN="$WORK_DIR/01-gem5/03-perf/no-prefetch"
mkdir -p "$RUN"
rm -rf "$RUN/m5out"
GEM5_BIN=$(get_asset gem5-precompile/gem5.fast)
# memcpy workload 的 bin 文件
MEMCPY=$(get_asset workload/gem5-memcpy-riscv64-xs.bin)
export LD_LIBRARY_PATH="$(get_asset gem5-precompile):${LD_LIBRARY_PATH:-}"
export GCBV_REF_SO=$(get_asset gem5-precompile/riscv64-nemu-interpreter-so)
cd "$GEM5_HOME"

# 与执行 hello 程序类似，只需要更换 --generic-rv-cpt 为对应的 workload bin 文件即可

# 使用 --no-pf 参数来关闭 prefetcher
"$GEM5_BIN" --outdir="$RUN/m5out" \
  "$GEM5_HOME/configs/example/kmhv3.py" \
  --raw-cpt --generic-rv-cpt="$MEMCPY" --no-pf \
  2>&1 | tee "$RUN/run.log" >/dev/null

cat "$RUN/run.log" | tail -n 20

从结果上可以看到，关闭预取的 kmhv3 GEM5 模型的 memcpy 搬运数据的速度在 5076.06 MiB/s 左右

## 3. Top-Down：先判断瓶颈属于哪一侧

Top-Down[1] 是一种自顶向下、逐层分解的微架构性能分析方法，可以回答宏观问题：执行时间主要消耗在前端、错误推测、乱序执行还是内存侧

[1]: Yasin A. A top-down method for performance analysis and counters architecture\[C\]//2014 IEEE International Symposium on Performance Analysis of Systems and Software (ISPASS). IEEE, 2014: 35-44.

XS-GEM5 性能计数器支持 Top-Down 功能，可以使用 `gem5_data_proc` 仓库下的脚本来处理 memcpy 执行后的输出目录，得到 Top-Down 原始数据

In [ ]:
%%bash
set -euo pipefail
source ../env.sh

RUN="$WORK_DIR/01-gem5/03-perf/no-prefetch"
INPUT="$RUN/topdown-input"
mkdir -p "$INPUT"
ln -sfn "$RUN/m5out" "$INPUT/current"

# 使用 gem5_data_proc 处理 topdown 数据
# -t --topdown-raw 参数指示 gem5_data_proc 处理 topdown 数据
export PYTHONPATH="$GEM5_DATA_PROC_HOME"
python3 "$GEM5_DATA_PROC_HOME/batch.py" \
  -s "$INPUT" -t --topdown-raw \
  -o "$RUN/topdown-raw.csv" \
  > "$RUN/topdown.log"

printf 'Top-Down data: %s\n' "$RUN/topdown-raw.csv"

这是一个 csv 文件，先查看一下里面的内容

In [ ]:
%%bash
set -euo pipefail
source ../env.sh

RUN="$WORK_DIR/01-gem5/03-perf/no-prefetch"
cat "$RUN/topdown-raw.csv"

运行以下脚本，将以上原始 topdown 数据进行加工，便于阅读

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "GEM5").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "01-gem5"))

from gem5_tutorial_utils import plot_topdown
fig, (level1, memory) = plot_topdown(
    ROOT / "work/01-gem5/03-perf/no-prefetch/topdown-raw.csv"
)
# display(level1.round(2).to_frame("percent"))
# display(memory.round(2).to_frame("percent"))
plt.show()

上图表明有 38% 是 Retiring，表示无阻塞正常退休，60.9% 集中在 Memory，进一步细分 Memory bound 可以看到 L1/L2/DRAM stall 分别占据了 22.7%/26.3%/45.2%。

因此 Top-Down 帮我们确认了：处理器的瓶颈来自于访存。

接下来我们使用 PerfCCT 工具来深入 CPU 内部分析。

## 4. PerfCCT：指令生命周期与热点循环提取

PerfCCT 将每一条指令的在乱序执行的各个生命周期时刻（例如：取值、译码、重命名、执行、写回等）和其他关键信息记录到数据库中，首先我们重新运行一遍 memcpy，这次开启 PerfCCT。

In [ ]:
%%bash
set -euo pipefail
source ../env.sh

RUN="$WORK_DIR/01-gem5/03-perf/no-prefetch-perfcct"
mkdir -p "$RUN"
rm -rf "$RUN/m5out"
rm -f "$RUN/perfcct.db"
GEM5_BIN=$(get_asset gem5-precompile/gem5.fast)
MEMCPY=$(get_asset workload/gem5-memcpy-riscv64-xs.bin)
export LD_LIBRARY_PATH="$(get_asset gem5-precompile):${LD_LIBRARY_PATH:-}"
export GCBV_REF_SO=$(get_asset gem5-precompile/riscv64-nemu-interpreter-so)
cd "$GEM5_HOME"

# 使用 --enable-arch-db --arch-db-dump-lifetime 开启 perfcct 数据库的生成
# 使用 --arch-db-file 指定 perfcct 数据库的输出路径
"$GEM5_BIN" --outdir="$RUN/m5out" \
  "$GEM5_HOME/configs/example/kmhv3.py" \
  --raw-cpt --generic-rv-cpt="$MEMCPY" --no-pf \
  --enable-arch-db --arch-db-dump-lifetime \
  --arch-db-file="$RUN/perfcct.db" \
  2>&1 | tee "$RUN/run.log" >/dev/null

cat "$RUN/run.log" | tail -n 20

运行结束，得到 PerfCCT 的数据库文件路径，XS-GEM5 提供了 `ClockAnalysis.py` 来处理 PerfCCT 的数据，我们先查看一下里面的原始数据

In [ ]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "GEM5").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "01-gem5"))

import gem5_tutorial_utils
gem5_tutorial_utils = importlib.reload(gem5_tutorial_utils)
from gem5_tutorial_utils import cpu_clock_period, hot_window, clock_analysis_perfcct

DB = ROOT / "work/01-gem5/03-perf/no-prefetch-perfcct/perfcct.db"
PERIOD = cpu_clock_period(ROOT / "work/01-gem5/03-perf/no-prefetch-perfcct/m5out/stats.txt")
START_PC, END_PC = 0x80000140, 0x80000192
start_cycle, end_cycle = hot_window(DB, START_PC, END_PC, PERIOD)

print(f"CPU clock period: {PERIOD} ticks/cycle")

# 查看 PerfCCT 数据库中指定范围的数据
clock_analysis_perfcct(
    ROOT / "GEM5/util/ClockAnalysis.py",
    DB,
    start_cycle,
    end_cycle,
    START_PC,
    END_PC,
    PERIOD,
    attention_pcs=(0x80000150, 0x80000166),
)

PerfCCT 数据库里面的每一条记录存储的是指令的 pc、反汇编和在乱序流水线各个 stage 的时刻信息。

接下来继续使用 `ClockAnalysis.py` 工具来导出热点基本块。工具按出现次数排序输出热点块，本次演示展示 memcpy top1 的热点基本块。

In [ ]:
%%bash
source ../env.sh

RUN="$WORK_DIR/01-gem5/03-perf/no-prefetch-perfcct"
DB="$RUN/perfcct.db"
PERIOD=$(awk '$1 == "system.cpu_clk_domain.clock" { print $2; exit }' \
  "$RUN/m5out/stats.txt")
OUTPUT="$RUN/clock-analysis-top1-inter-gap.txt"

# --period 指定 CPU 时钟周期，XS-GEM5 中 333 tick 为一个 CPU 时钟周期
# 用 `--tool bbl --inter-gap` 在完整动态指令 trace 上重建 basic block
python3 \
  "$GEM5_HOME/util/ClockAnalysis.py" \
  "$DB" \
  --period "$PERIOD" \
  --tool bbl \
  --inter-gap \
  2>/dev/null \
| awk '
/^Count:/ {
    count++
    if (count == 2) exit
}
{ print }
' \
| tee "$OUTPUT"

printf '\nExported top1 inter-gap: %s\n' "$OUTPUT"

以上结果就是 memcpy 的 top1 热点基本块：19 条指令组成的循环展开版本的内存拷贝（符合我们对 memcpy 的认知），在完整 trace 中重复执行了 8191 次。fetch decode 这些数值当前指令和上一条指令到达对应 stage 的平均时间差，先不用关心。

接下来我们继续使用 `ClockAnalysis.py` 工具提取热点基本块某一次执行时的指令执行时间轴

In [ ]:
from gem5_tutorial_utils import plot_perfcct_timeline

# 使用 perfcct 数据库绘制指定 PC 范围内的时钟周期的可视化时间轴
fig, timeline = plot_perfcct_timeline(
    DB,
    START_PC,
    END_PC,
    period=PERIOD,
    start_cycle=start_cycle,
    end_cycle=end_cycle,
)
plt.show()

可以观察到 load store 指令的执行延迟极高，与 Topdown 中的结果呼应。

导致此现象的原因：没有打开预取，指令总是发生 cache miss，总是需要从 DDR 中获取数据，带来高额的 miss latency 开销。

预取是一个很好的解决内存墙问题的技术，我们在 kmhv3 上也进行了大量的预取优化，接下来看看开启 kmhv3 预取后的情形。

## 5. 开启 KMHv3 默认预取

In [ ]:
%%bash
set -euo pipefail
source ../env.sh

RUN="$WORK_DIR/01-gem5/03-perf/kmhv3-default"
mkdir -p "$RUN"
rm -rf "$RUN/m5out"
rm -f "$RUN/perfcct.db"
GEM5_BIN=$(get_asset gem5-precompile/gem5.fast)
MEMCPY=$(get_asset workload/gem5-memcpy-riscv64-xs.bin)
export LD_LIBRARY_PATH="$(get_asset gem5-precompile):${LD_LIBRARY_PATH:-}"
export GCBV_REF_SO=$(get_asset gem5-precompile/riscv64-nemu-interpreter-so)
cd "$GEM5_HOME"

# 去掉 --no-pf 参数，就可以使用默认的开启预取版本的 kmhv3
"$GEM5_BIN" --outdir="$RUN/m5out" \
  "$GEM5_HOME/configs/example/kmhv3.py" \
  --raw-cpt --generic-rv-cpt="$MEMCPY" \
  --enable-arch-db --arch-db-dump-lifetime \
  --arch-db-file="$RUN/perfcct.db" \
  2>&1 | tee "$RUN/run.log" >/dev/null

cat "$RUN/run.log" | tail -n 20

从结果上看，开启预取后的带宽来到了 23977.37 MiB/s。

### 开启预取与关闭预取的 memcpy 带宽对比

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "GEM5").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "01-gem5"))

from gem5_tutorial_utils import plot_memcpy_bandwidth
fig, results = plot_memcpy_bandwidth({
    "No prefetch": ROOT / "work/01-gem5/03-perf/no-prefetch/run.log",
    "KMHv3 default": ROOT / "work/01-gem5/03-perf/kmhv3-default/run.log",
})
# display(results[["cycles", "payload_mib_s", "traffic_mib_s", "speedup"]])
plt.show()

开启预取后 memcpy 的带宽获得了 4.72 倍的提升。

接下来，我们看看为什么开启预取能带来这么显著的效果，通过观察 load 访存指令的执行延迟的变化也许可以找到原因。

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "GEM5").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "01-gem5"))

from gem5_tutorial_utils import cpu_clock_period, plot_load_gap_comparison

PERIOD = cpu_clock_period(ROOT / "work/01-gem5/03-perf/no-prefetch-perfcct/m5out/stats.txt")
fig, gaps = plot_load_gap_comparison({
    "No prefetch": ROOT / "work/01-gem5/03-perf/no-prefetch-perfcct/perfcct.db",
    "KMHv3 default": ROOT / "work/01-gem5/03-perf/kmhv3-default/perfcct.db",
}, 0x80000140, 0x80000192, period=PERIOD, pc_count=8)
plt.show()

结果是符合预期的，预取开启后，数据会被提早预取到 cache 中，load 的执行延迟会先缩短，流水线提交停顿随之减少，最终反映为更高的 memcpy 带宽。

### 开启预取后的 Instruction timeline

最后我们看一下开启预取后的热点基本块某次执行时的指令 timeline。

In [ ]:
from gem5_tutorial_utils import plot_perfcct_timeline

fig, _ = plot_perfcct_timeline(
    ROOT / "work/01-gem5/03-perf/kmhv3-default/perfcct.db",
    START_PC,
    END_PC,
)
fig.axes[0].set_title("PerfCCT instruction timeline: KMHv3 default prefetch")
plt.show()

与关闭预取时的 timeline 相比，横轴周期数明显减少，load 的执行延迟也显著缩短，印证了预取的作用。

本节从微架构优化的角度对 memcpy 进行了程序特性分析和微架构瓶颈定位，也看到了 kmhv3 预取器的显著优化效果，下一节我们将尝试添加一个自己设计的预取器到 XS-GEM5 中。